# Week 4 exercise answers: combining Scottish health datasets

These are worked solutions. Other clear approaches can also be correct.

The comments explain both what the code does and why each step is needed.

## Setup

**Question:** Load the five prepared CSV files as DataFrames.

In [ ]:
import pandas as pd
from IPython.display import display

# Each variable ends in _df to remind readers that it contains a DataFrame.
referrals_2024_df = pd.read_csv("therapy_referrals_2024.csv")
referrals_2025_df = pd.read_csv("therapy_referrals_2025.csv")
waiting_df = pd.read_csv("therapy_waiting_2025.csv")
health_boards_df = pd.read_csv("health_boards.csv")
population_df = pd.read_csv("health_board_population_2025.csv")

## Inspect the tables

**Question:** Inspect every DataFrame. What does one row represent, and which
columns can connect the tables?

In [ ]:
# head() shows example records; shape reports (rows, columns).
for label, table_df in [
    ("Referrals 2024", referrals_2024_df),
    ("Referrals 2025", referrals_2025_df),
    ("Waiting 2025", waiting_df),
    ("Health boards", health_boards_df),
    ("Population", population_df),
]:
    print(label, table_df.shape)
    print(list(table_df.columns))
    display(table_df.head())

**Interpretation:**

- A referrals row describes referrals for one reporting category in one month.
- A waiting row describes the waiting list for one reporting category in one month.
- A health-board row connects one territorial `HB` code to its readable name.
- A population row gives the 2025 population of one territorial health board.

`HB` connects all four kinds of table. The referrals and waiting tables also need
`Month`, because each health-board code occurs in several months.

## Concatenate two years of referrals

**Question:** Stack the 2024 and 2025 referrals files. Does the combined row count
equal the two original counts added together?

In [ ]:
# concat() stacks tables that have the same columns.
# ignore_index=True creates one fresh index instead of retaining two old indices.
referrals_df = pd.concat(
    [referrals_2024_df, referrals_2025_df],
    ignore_index=True,
)

print("2024 rows:", len(referrals_2024_df))
print("2025 rows:", len(referrals_2025_df))
print("Combined rows:", len(referrals_df))

# head() and tail() let us check both ends of the combined table.
display(referrals_df.head())
display(referrals_df.tail())

**Interpretation:** The combined table contains 393 rows, exactly
192 + 201. The first records are from January 2024 and the final records are from
December 2025.

## Merge referrals with waiting information

**Question:** Merge the 2025 referrals and waiting tables using `HB` and `Month`.
Why are both columns required?

In [ ]:
# A match requires the same reporting category AND the same month.
# Passing a list to on= tells pandas to use a two-column key.
therapy_df = pd.merge(
    referrals_2025_df,
    waiting_df,
    on=["HB", "Month"],
    how="inner",
)

print("Referrals rows before merge:", len(referrals_2025_df))
print("Rows after merge:", len(therapy_df))
display(therapy_df.head())

**Interpretation:** The merge retains 201 rows, so every 2025 referral
record found one waiting-list record with the same `HB` and `Month`. Using only
`HB` would match each board's January referrals to that board's February, March
and every other month as well.

## Add readable health-board names

**Question:** Left-join the names lookup. Which December codes do not receive a
territorial health-board name?

In [ ]:
# A left join keeps every therapy row, even when the lookup has no match.
therapy_named_df = pd.merge(
    therapy_df,
    health_boards_df,
    on="HB",
    how="left",
)

# isna() is True where the merge could not add a board name.
print(therapy_named_df["HealthBoard"].isna().sum(), "rows have no board name")

# Build simple questions separately before combining them.
select_december = therapy_named_df["Month"] == 202512
select_missing_name = therapy_named_df["HealthBoard"].isna()
select_rows = select_december & select_missing_name

therapy_named_df.loc[
    select_rows,
    ["HB", "Month", "ReferralsReceived", "HealthBoard"],
]

**Interpretation:** `S92000003`, `SB0801` and `SB0803` are not among
the 14 territorial health boards. `S92000003` is the Scotland total; the `SB`
codes are special national boards. We inspect unmatched rows because a missing
match can represent a meaningful category, a changed code or a genuine data error.

## Keep the territorial health boards

**Question:** Inner-join the original therapy data with the health-board lookup.
What happens to unmatched categories, and why does each board still occur many
times?

In [ ]:
# An inner join retains only HB values present in both tables.
# Starting again from therapy_df avoids merging the HealthBoard column twice.
territorial_df = pd.merge(
    therapy_df,
    health_boards_df,
    on="HB",
    how="inner",
)

print("Rows before territorial-board match:", len(therapy_df))
print("Rows after territorial-board match:", len(territorial_df))

**Interpretation:** The Scotland total and two special-board categories
are removed, leaving 168 rows. Each territorial board still occurs repeatedly
because there is one row for every month, not one row for every board.

## Add population information

**Question:** Add each board's 2025 population. Why does its population repeat
across several rows, and what type of merge is this?

In [ ]:
# Many monthly therapy rows match one population row for the same board.
# This is therefore a many-to-one merge.
territorial_df = pd.merge(
    territorial_df,
    population_df,
    on="HB",
    how="left",
)

print("Merged shape:", territorial_df.shape)
print("Missing population values:", territorial_df["Population"].isna().sum())
display(territorial_df.head())

**Interpretation:** Population is a property of the board, so the same
value is added to every monthly row belonging to that board. No territorial board
is missing its population.

## Compare health boards in December 2025

**Question:** Compare December boards by waiting count and population. What stands
out, and why are raw counts not sufficient for a fair comparison?

In [ ]:
# Select one month so that each territorial board contributes one row.
select_december = territorial_df["Month"] == 202512
display_columns = [
    "HealthBoard",
    "Population",
    "ReferralsReceived",
    "TotalPatientsWaiting",
    "WaitingOver52Weeks",
]
december_df = territorial_df.loc[select_december, display_columns]

# First examine the boards with the largest waiting lists.
december_df = december_df.sort_values("TotalPatientsWaiting", ascending=False)
print("Sorted by people waiting")
display(december_df)

# Reuse the same DataFrame and change only its ordering.
december_df = december_df.sort_values("Population", ascending=False)
print("Sorted by population")
display(december_df)

**Interpretation:** NHS Lothian has the largest December waiting count,
while NHS Greater Glasgow and Clyde has the largest population and the most
referrals received. Larger boards often have larger raw counts, but the ordering
is not identical.

Raw counts do not account for the number of people served by each board. The merge
has supplied the missing population denominator. In the next stage of the course,
that denominator could be used to derive a rate per 100,000 people.

## Stretch debugging: the merge that quietly multiplies rows

**Question:** Why does merging only on `HB` create too many rows? Repair it.

In [ ]:
# This code runs, but HB is not enough to identify a monthly observation.
# Every month in one table matches every month for the same HB in the other table.
wrong_df = pd.merge(
    referrals_2025_df,
    waiting_df,
    on="HB",
)

print("Wrong shape:", wrong_df.shape)
display(wrong_df[["HB", "Month_x", "Month_y"]].head(15))

In [ ]:
# Include Month so January matches January, February matches February, etc.
corrected_df = pd.merge(
    referrals_2025_df,
    waiting_df,
    on=["HB", "Month"],
)

print("Corrected shape:", corrected_df.shape)
corrected_df.head()

The incorrect merge creates 2,385 rows because it makes many-to-many
matches between months within each `HB`. The corrected two-column key returns the
expected 201 monthly observations. This is a particularly dangerous error because
the original code runs without complaining.

## Data sources

- [Psychological Therapies Waiting Times](https://www.opendata.nhs.scot/dataset/psychological-therapies-waiting-times)
- [Scottish population estimates](https://www.opendata.nhs.scot/dataset/population-estimates)
- [Health Board codes and labels](https://www.opendata.nhs.scot/dataset/geography-codes-and-labels)

The classroom files contain selected rows and columns from these official datasets.